In [ ]:
"""
Filtering and exploration of the data; find good FIPS/sewersheds to use for the analysis.
Criteria are (roughly) good quality liquid_perc data going back to 2021, sizeable population,
and a reasonable visual fit to the JHU case data.
"""

import pandas as pd
import numpy as np

from datetime import datetime

# Represent numpy data as plain text, numpy2.x is a bit verbose for interactive use
np.set_printoptions(legacy="1.13")

# Default to plotly for interactive visualizations
pd.options.plotting.backend = "plotly"

from wbe.constants import DATA_PATH
from wbe.inputs import split_concentration_var, group_data, get_cdc_wbe_data

from wbe.inputs import get_jhu_county_data


In [ ]:
filename = "cdc_data_d20260225_t222606_sha6fc34cc.csv"
data = pd.read_csv(DATA_PATH / "wbe" / filename, parse_dates=["sample_collect_date"])

data = split_concentration_var(data)

liquid_data = data[data["liquid_pcr_conc"].notna()].copy()
solid_data = data[data["solid_pcr_conc"].notna()].copy()

liquid_obs = group_data("liquid", liquid_data)
solid_obs = group_data("solid", solid_data)

In [ ]:
death_data = get_jhu_county_data("jhu_deaths_d20260226_t065541_sha8e7d05f.csv").diff().clip(lower=0)
case_data = get_jhu_county_data("jhu_confirmed_d20260226_t065534_sha8e7d05f.csv").diff().clip(lower=0)

In [ ]:
class ObsProvider:
    def __init__(self, data):
        self.data = data
        self.fips = data["fips"].unique()
        self.sewershed_id = data["sewershed_id"].unique()

    def subset_by(self, col, val):
        return self.data[self.data[col]==val]


In [ ]:
op = ObsProvider(liquid_obs)

In [ ]:
start_time_fips = pd.Series({fip:op.subset_by("fips", fip).index.min() for fip in op.fips})
start_time_sewershed = pd.Series({s:op.subset_by("sewershed_id", s).index.min() for s in op.sewershed_id})

In [ ]:
# Start by just grabbing FIPS series that start before mid-2020, there aren't many

LATEST_START = datetime(2020, 6, 1)

start_time_fips[start_time_fips <= LATEST_START]

In [ ]:
FIPS_CODE = "06037"
# This seems like a good candidate; only 2 sewersheds (and only one that really has any data to speak of)
op.subset_by("fips", FIPS_CODE).pivot(columns=["sewershed_id"], values="pcr_conc").plot()

In [ ]:
op.subset_by("fips", FIPS_CODE).pivot(columns=["sewershed_id"], values="pcr_conc")[114].loc["jul 2020":"jul 2022"].plot()

In [ ]:
death_data[FIPS_CODE].loc["jul 2020":"jul 2022"].rolling(7).mean().plot()

In [ ]:
case_data[FIPS_CODE].loc["jul 2020":"jul 2022"].rolling(7).mean().plot()

In [ ]:
# Some have many sewersheds with disparate data quality, so not ideal starting points (we can come back to these later)
op.subset_by("fips", "04013").pivot(columns=["sewershed_id"], values="pcr_conc").plot()